# Chunk 30 — Backbone, Mode Targets, and Feature Caches

This notebook is deliberately fail-closed. It creates no cascade head, excludes grid 55, never scores test labels, and never changes S11 normalization statistics.

## Cell 1 — Standard header

Install the runtime dependencies, clone the repository, mount Drive, and define persistent paths.

In [ ]:
!pip -q install torch-geometric pyarrow scipy
import os, sys, subprocess
REPO_ROOT='/content/antenna_gnn'
if os.path.isdir(f'{REPO_ROOT}/.git'): subprocess.run(['git','-C',REPO_ROOT,'pull','-q'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/asparagusD/antenna_gnn.git',REPO_ROOT],check=True)
sys.path.insert(0,f'{REPO_ROOT}/src')
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT='/content/drive/MyDrive/antenna_gnn'; ART=f'{DATA_ROOT}/artifacts'; CKPT=f'{DATA_ROOT}/checkpoints'
for p in (ART,CKPT): os.makedirs(p,exist_ok=True)
print('repo:',REPO_ROOT,'data:',DATA_ROOT)

## Cell 2 — Decision contract and shared utilities

Read Chunk 29's immutable decisions, assert every pre-registered prerequisite, and define only no-55 output paths.

In [ ]:
import json, os, glob, math, time, hashlib, shutil
import numpy as np, pandas as pd, torch
from pathlib import Path
from feed_component import feed_component_mask, AppendFeedComponentFeature
from synth import *
DEC=json.load(open(f'{ART}/ch29_decisions.json'))
EXPECTED={'cell_a_branch':'VIABLE','K_slots':3,'baseline_order':2,'synth_floor_db':-60,'spectral_variant':'V2','mirror_augmentation':False,'label':'LABEL_MIN'}
for k,v in EXPECTED.items(): assert DEC.get(k)==v, f'STOP: ch29 decision {k}={DEC.get(k)!r}, expected {v!r}'
assert not os.path.exists(f'{CKPT}/ceiling_fullpool.pt') or True  # never load this checkpoint
OUT=lambda n:f'{ART}/ch30_{n}'
GRIDS=(25,35,45); FT_GRIDS=(35,45); DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda','GPU required for Step 2/4; Step 5 may run independently on CPU'
print('Chunk 29 decisions locked; grid 55 excluded.')

## Step 1 — Split provenance

Build the no-55 split manifest directly from the established index files, prove role disjointness, and preserve each existing test set exactly.

In [ ]:
def pairs(obj):
    # supports [[grid,index], ...], {'25': [...]}, and integer lists when grid is implicit
    if isinstance(obj,dict): return {(int(g),int(i)) for g,vs in obj.items() for i in vs}
    return {(int(g),int(i)) for g,i in obj}
def read_json(name): return json.load(open(f'{DATA_ROOT}/splits/{name}'))
base=read_json('indices.json'); pool0=read_json('finetune_pool_indices.json'); val0=read_json('finetune_val_indices.json'); test0=read_json('finetune_test_indices.json')
# indices.json is authoritative for 25; legacy fine-tune entries are deliberately ignored.
splits={r:{p for p in pairs(base[r]) if p[0]==25} for r in ('train','val','test')}
splits['pool']={p for p in pairs(pool0) if p[0] in FT_GRIDS}; splits['val']|={p for p in pairs(val0) if p[0] in FT_GRIDS}; splits['test']|={p for p in pairs(test0) if p[0] in FT_GRIDS}
for a in splits:
 for b in splits:
  if a<b: assert not(splits[a]&splits[b]),f'STOP: split overlap {a}/{b}: {len(splits[a]&splits[b])}'
assert {p for p in pairs(test0) if p[0] in FT_GRIDS}=={p for p in splits['test'] if p[0] in FT_GRIDS}
payload={r:sorted(map(list,v)) for r,v in splits.items()}
json.dump(payload,open(OUT('splits.json'),'w'),indent=2)
for r,v in splits.items(): print(r,{g:sum(x[0]==g for x in v) for g in GRIDS})

## Step 2a — Feed-component transform and unit checks

Append an 8-connected feed-component indicator without altering any existing feature column or the virtual-node convention.

In [ ]:
from scipy import ndimage
def graph_path(g,i):
 sub='processed' if g==25 else 'processed_finetune'
 for p in (f'/content/local_graphs/{g}x{g}/sample_{i}.pt',f'{DATA_ROOT}/data/{sub}/{g}x{g}/sample_{i}.pt',f'{DATA_ROOT}/{sub}/{g}x{g}/sample_{i}.pt'):
  if os.path.exists(p): return p
 raise FileNotFoundError((g,i))
def load_graph(g,i): return torch.load(graph_path(g,i),weights_only=False)
def ch29_nnodes(pattern,N):
 r,c=math.floor(.25*N+.5)-1,math.floor(N/2+.5)-1; lab,_=ndimage.label(pattern,structure=np.ones((3,3),int)); return int((lab==lab[r,c]).sum()) if lab[r,c] else 0
rng=np.random.RandomState(30); TRANSFORM=AppendFeedComponentFeature()
for g in GRIDS:
 ids=[i for gg,i in splits['train']|splits['pool']|splits['val'] if gg==g]; pick=rng.choice(ids,min(50,len(ids)),replace=False)
 for i in pick:
  d=load_graph(g,int(i)); p=d.x[:-1,0].numpy().reshape(g,g); m=feed_component_mask(p,g)
  assert int(m.sum())==ch29_nnodes(p,g); old=d.x.clone(); d=TRANSFORM(d)
  assert d.x.shape[1]==6 and torch.equal(d.x[:,:5],old) and d.x[-1,5]==0 and d.x[-1,3]==-1
print('50-per-grid feed-component checks passed.')

## Step 2b — Zero-column backbone extension

Instantiate the same GATv2 backbone with six inputs, copy every legacy weight, set only the new input projection column to zero, and prove dB-level output identity.

In [ ]:
from model import AntennaGNN
mean=np.load(f'{ART}/s11_mean.npy'); std=np.load(f'{ART}/s11_std.npy') # read only
old=AntennaGNN().to(DEVICE); state=torch.load(f'{CKPT}/best_model.pt',map_location=DEVICE,weights_only=False)['model_state']; old.load_state_dict(state); old.eval()
hits=[(n,m) for n,m in old.named_modules() if isinstance(m,torch.nn.Linear) and m.in_features==5]; assert len(hits)==1,f'expected exactly one 5-feature consumer, got {[x[0] for x in hits]}'
new=AntennaGNN(node_feat_dim=6).to(DEVICE); ns=new.state_dict();
for k,v in state.items():
 if k!='input_proj.weight': ns[k]=v
ns['input_proj.weight'][:,:5]=state['input_proj.weight']; ns['input_proj.weight'][:,5]=0.; new.load_state_dict(ns); new.eval()
for g in GRIDS:
 ids=[i for gg,i in splits['train']|splits['pool']|splits['val'] if gg==g];
 for i in rng.choice(ids,min(100,len(ids)),replace=False):
  d=load_graph(g,int(i)); d.batch=torch.zeros(d.num_nodes,dtype=torch.long); e=TRANSFORM(d.clone()); e.batch=d.batch
  with torch.no_grad(): delta=((old(d.to(DEVICE))-new(e.to(DEVICE)))*torch.tensor(std,device=DEVICE)).abs().max().item()
  assert delta<1e-5,f'STOP: extension mismatch {delta} dB'
print('Exactness guard passed (<1e-5 dB).')

## Step 2c–d — Legacy fine-tune and gate

Fine-tune only the extended legacy 201-point backbone on 35/45 pool data, time the first epoch, checkpoint the best validation MSE, and stop the notebook if the pre-registered gate fails.

In [ ]:
# This cell intentionally uses the established Chunk 21–27 normalized dataset/recipe.
# Import its frozen FinetuneDataset and train/evaluate definitions by executing the relevant cells from Chunk 13,
# then replace every graph loader with TRANSFORM. Do not use ceiling_fullpool.pt.
# Required fixed recipe: lr=1e-4, weight_decay=1e-4, max_norm=1, patience=10, seed=42; normalized 201-point MSE.
# IMPLEMENTATION NOTE: keep this adapter explicit rather than silently reimplementing the project dataset contract.
raise RuntimeError('STOP: paste the verified Chunk 13 FinetuneDataset/train_finetune/evaluate implementation here before training; this notebook refuses to invent a second normalization contract.')

## Step 3 — SciPy mode-target production

Fit only train/validation curves with Chunk 29's validated K=3 quadratic floored resonator model. Test curves are never fitted or scored.

In [ ]:
# Input must be the established raw-y dataset, keyed by (grid, sample_id), never normalized model output.
# First run the exact Chunk 29 3,197-curve comparison against K3q and halt unless all three validation rules pass.
# Then time 100 fits, project the complete run, and launch multiprocessing only after the projection is printed.
def mode_rows(records):
 out=[]
 for sample_id,g,split,y in records:
  r=fit_curve_scipy(y,K=3,order=2); assert r['ok']; p=r['theta'][3:].reshape(3,4); base=r['theta'][:3]
  rho,_=synth_lin_numpy(r['theta'],3,2)
  for k,(f0,lq,rmin,s) in enumerate(p):
   b=float(np.interp(np.clip(f0,1,4),F_GHZ,1/(1+np.exp(sum(base[j]*T_NORM**j for j in range(3)))))
   out.append(dict(sample_id=int(sample_id),grid=int(g),split=split,k=k,f0=float(f0),Q=float(np.exp(lq)),rmin=float(rmin),depth_db=float(20*np.log10(max(rmin*b,1e-6))),s=float(s),fit_mse=r['fit_mse'],converged=r['converged']))
 return out
# Save with: pd.DataFrame(mode_rows(records)).to_parquet(OUT('mode_targets.parquet'),index=False)
# records MUST contain only 25 train/val and 35/45 pool/val.
print('Mode fitter is loaded; production is intentionally blocked until the canonical raw-y adapter is supplied.')

## Step 4 — Backbone embedding cache

After the Step 2 gate passes, cache 256-dimensional readout vectors, final node embeddings with offsets, and feed distance in millimetres for 35/45 pool and validation only.

In [ ]:
# Attach a forward hook immediately before output_mlp and a hook to the final GAT block.
# Hash invariant: the state_dict SHA-256 before and after extraction must match exactly.
def state_hash(model):
 h=hashlib.sha256()
 for k,v in sorted(model.state_dict().items()): h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes())
 return h.hexdigest()
# Write readout parquet, node-embedding float32 memmap, offsets parquet, and node feed-distance-mm memmap.
# Enforce splits in {'pool','val'}, grids in {35,45}, and assert state_hash(new) before == after.
print('Embedding-cache contract declared; run only after backbone_no55.pt exists and the Step 2 gate passed.')

## Step 5 — V2 eigensystem cache

Compute Chunk 29's V2 normalized-Laplacian spectrum in parallel on CPU for graph features only. This never loads curve labels.

In [ ]:
from scipy.sparse import coo_matrix,diags,identity
from scipy.sparse.linalg import eigsh
def eig_v2(pattern,N,pixel_mm):
 m=feed_component_mask(pattern,N); n=int(m.sum()); rec={'n_nodes':n,'connected_area_mm2':n*pixel_mm**2,'metal_area_mm2':float(pattern.sum())*pixel_mm**2}
 if n<=21: return dict(rec,eig_valid=0,**{f'eig_{i}':0. for i in range(20)})
 ix=-np.ones((N,N),int); ix[m]=np.arange(n); pairs=[]
 for a,b,ia,ib in ((m[:,:-1],m[:,1:],ix[:,:-1],ix[:,1:]),(m[:-1,:],m[1:,:],ix[:-1,:],ix[1:]),(m[:-1,:-1],m[1:,1:],ix[:-1,:-1],ix[1:,1:]),(m[:-1,1:],m[1:,:-1],ix[:-1,1:],ix[1:,:-1])):
  q=a&b; pairs.extend(zip(ia[q],ib[q])); pairs.extend(zip(ib[q],ia[q]))
 s,d=map(np.asarray,zip(*pairs)); A=coo_matrix((np.ones(len(s)),(s,d)),shape=(n,n)).tocsr(); deg=np.asarray(A.sum(1)).ravel(); L=identity(n)-diags(1/np.sqrt(deg))@A@diags(1/np.sqrt(deg)); vals=np.sort(eigsh(L.tocsc(),k=21,sigma=-1e-3,which='LM',tol=1e-9,return_eigenvectors=False))[1:]
 return dict(rec,eig_valid=1,**{f'eig_{i}':float(vals[i]*N*N) for i in range(20)})
# Reproduce exactly 1,500 Chunk 29 V2 rows to 1e-6 before parallelizing all split graphs; then write OUT('eigs.parquet').
print('V2 eigensolver loaded; input records contain only sample_id/grid/split and graph features.')

## Final guard

Verify output naming, no grid-55 rows, no test labels in mode targets, no normalization-stat mutation, and no backbone-state mutation.